# Parking Birmingham — Data Audit and Validation

**Course:** ADSP 32021 — Machine Learning Operations  
**Project:** Parking Availability Prediction Using an Automated MLOps Pipeline

## 1. Introduction

This notebook performs an initial audit of the UCI Parking Birmingham dataset to assess data quality, structure, and suitability for forecasting parking occupancy.

The audit will examine:
- Dataset dimensions, column names, and data types
- Missing values and duplicate observations
- Timestamp coverage and observation frequency
- Parking capacity and occupancy constraints
- Feasibility of a 30-minute prediction horizon

The findings will inform preprocessing, feature engineering, and chronological data splitting.

**Data source:** [UCI Parking Birmingham](https://archive.ics.uci.edu/dataset/482/parking+birmingham)


## 2. Dataset Loading and Initial Inspection

We begin by loading the raw dataset and examining its dimensions, column names, data types, and sample observations.

This initial inspection helps identify potential formatting issues and establishes a baseline understanding of the available variables before applying any transformations.

In [1]:
import pandas as pd
from pathlib import Path

# Path to the raw dataset
data_path = Path("../data/dataset.csv")

# Load the dataset
df = pd.read_csv(data_path)

# Inspect dataset structure
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nData types:")
print(df.dtypes)

# Display the first five observations
display(df.head())


Dataset shape: (35717, 4)

Column names: ['SystemCodeNumber', 'Capacity', 'Occupancy', 'LastUpdated']

Data types:
SystemCodeNumber    object
Capacity             int64
Occupancy            int64
LastUpdated         object
dtype: object


,SystemCodeNumber,Capacity,Occupancy,LastUpdated
0,BHMBCCMKT01,577,61,2016-10-04 07:59:42
1,BHMBCCMKT01,577,64,2016-10-04 08:25:42
2,BHMBCCMKT01,577,80,2016-10-04 08:59:42
3,BHMBCCMKT01,577,107,2016-10-04 09:32:46
4,BHMBCCMKT01,577,150,2016-10-04 09:59:48


### 2.1 Initial Inspection Findings

The dataset contains **35,717 observations and 4 variables**:

- **SystemCodeNumber (object):** Unique identifier for each parking facility.
- **Capacity (int64):** Total number of parking spaces at a facility.
- **Occupancy (int64):** Number of occupied parking spaces at the recorded time.
- **LastUpdated (object):** Timestamp indicating when the observation was recorded.

**Key observations:**

1. The dataset contains the expected number of observations and variables.
2. Capacity and Occupancy are already stored as integers, making them suitable for numerical analysis.
3. LastUpdated is currently stored as an object rather than a datetime type and will require conversion.
4. SystemCodeNumber is a categorical identifier and should not be treated as a continuous numerical variable.

**Next steps:** Evaluate missing values, duplicate records, timestamp consistency, and occupancy constraints before preprocessing.

## 3. Missing Values and Duplicate Records

Missing values can affect model training and feature engineering, particularly when constructing time-dependent features.

Duplicate observations may distort historical parking patterns or create misleading training examples.

We examine missing values across all variables, count exact duplicate rows, and check for repeated facility-timestamp combinations.


In [2]:
# Check missing values
print("Missing values by column:")
print(df.isnull().sum())

print("\nMissing value percentages:")
print((df.isnull().mean() * 100).round(2))

# Check duplicate records
print("\nExact duplicate rows:", df.duplicated().sum())

# Check duplicate facility-timestamp combinations
duplicate_keys = df.duplicated(
    subset=["SystemCodeNumber", "LastUpdated"]
).sum()

print("Duplicate facility-timestamp combinations:", duplicate_keys)

# Count unique parking facilities
print("\nUnique parking facilities:", df["SystemCodeNumber"].nunique())


Missing values by column:
SystemCodeNumber    0
Capacity            0
Occupancy           0
LastUpdated         0
dtype: int64

Missing value percentages:
SystemCodeNumber    0.0
Capacity            0.0
Occupancy           0.0
LastUpdated         0.0
dtype: float64

Exact duplicate rows: 216
Duplicate facility-timestamp combinations: 216

Unique parking facilities: 30


### 3.1 Missing Values and Duplicate Records — Findings

The dataset contains **35,717 observations across 30 parking facilities**.

**Missing values:**
- No missing values were identified in any of the four variables.
- No missing-value imputation is required for the original dataset.

**Duplicate records:**
- 216 exact duplicate rows were identified.
- 216 duplicate facility-timestamp combinations were identified.
- These duplicates may distort observed parking patterns and introduce redundant training examples.

**Cleaning decision:**

Exact duplicate records will be removed during preprocessing, retaining one observation per unique record. The expected cleaned dataset contains 35,501 observations.

The original CSV will remain unchanged to preserve reproducibility. Duplicate removal will be implemented in the preprocessing pipeline so the same transformation can be applied automatically during future pipeline executions.


## 4. Initial Data Cleaning

Exact duplicate observations are removed to prevent redundant records from affecting model development.

The raw dataset is preserved, and cleaning is performed on a separate DataFrame to maintain reproducibility and traceability.

Additional validation, including timestamp consistency and occupancy constraints, will be performed before the dataset is considered ready for modeling.


In [3]:
# Create a cleaned copy of the dataset
df_clean = df.drop_duplicates().copy()

# Verify duplicate removal
print("Original observations:", len(df))
print("Cleaned observations:", len(df_clean))
print("Removed duplicates:", len(df) - len(df_clean))

print("\nRemaining exact duplicates:",
      df_clean.duplicated().sum())

print("Remaining duplicate facility-timestamp combinations:",
      df_clean.duplicated(
          subset=["SystemCodeNumber", "LastUpdated"]
      ).sum())


Original observations: 35717
Cleaned observations: 35501
Removed duplicates: 216

Remaining exact duplicates: 0
Remaining duplicate facility-timestamp combinations: 0


### 4.1 Duplicate Removal Results

After removing exact duplicates, the dataset contains **35,501 observations**.

- **Original observations:** 35,717
- **Duplicate records removed:** 216
- **Remaining exact duplicates:** 0
- **Remaining duplicate facility-timestamp combinations:** 0

The cleaned dataset will be used for subsequent validation and exploratory analysis. The original dataset remains unchanged.

## 5. Timestamp Validation

Accurate timestamps are essential for time-series forecasting because observations must be ordered chronologically and historical features must not contain future information.

This section examines:
- Timestamp conversion and parsing errors
- Earliest and latest observation dates
- Observation intervals within each parking facility
- Whether the data supports a 30-minute forecasting horizon

Timestamp validation will guide the selection of an appropriate prediction horizon and the construction of leakage-safe lag features.


In [4]:
# Convert timestamps to datetime
df_clean["LastUpdated"] = pd.to_datetime(
    df_clean["LastUpdated"],
    errors="coerce"
)

# Check for invalid timestamps
print("Invalid timestamps:",
      df_clean["LastUpdated"].isna().sum())

# Check dataset date range
print("\nEarliest timestamp:",
      df_clean["LastUpdated"].min())

print("Latest timestamp:",
      df_clean["LastUpdated"].max())

# Sort observations chronologically within each facility
df_clean = df_clean.sort_values(
    ["SystemCodeNumber", "LastUpdated"]
).reset_index(drop=True)

# Calculate time between consecutive observations per facility
df_clean["TimeDifferenceMinutes"] = (
    df_clean.groupby("SystemCodeNumber")["LastUpdated"]
    .diff()
    .dt.total_seconds() / 60
)

# Summarize observation intervals
print("\nObservation interval statistics (minutes):")
print(df_clean["TimeDifferenceMinutes"].describe())

# Most common observation intervals
print("\nMost common observation intervals (rounded minutes):")
print(
    df_clean["TimeDifferenceMinutes"]
    .round()
    .value_counts()
    .head(10)
)


Invalid timestamps: 0

Earliest timestamp: 2016-10-04 07:46:28
Latest timestamp: 2016-12-19 16:30:35

Observation interval statistics (minutes):
count    35471.000000
mean        88.607939
std        322.241946
min          5.966667
25%         26.983333
50%         32.933333
75%         33.066667
max      13908.783333
Name: TimeDifferenceMinutes, dtype: float64

Most common observation intervals (rounded minutes):
TimeDifferenceMinutes
27.0     10148
33.0     10044
26.0      5276
34.0      5226
30.0       601
930.0      456
20.0       429
40.0       414
933.0      308
31.0       256
Name: count, dtype: int64


### 5.1 Timestamp Validation Findings

All 35,501 cleaned observations have valid timestamps. The dataset covers October 4 through December 19, 2016.

**Observation frequency:**
- Median interval: 32.93 minutes
- Mean interval: 88.61 minutes
- Minimum interval: 5.97 minutes
- Maximum interval: approximately 9.66 days

The most common observation intervals are approximately 26–27 and 33–34 minutes. This indicates that observations are irregularly spaced rather than recorded at exact 30-minute intervals.

Substantial gaps of approximately 930 minutes are also present. These may represent overnight periods without observations, although their cause has not yet been confirmed.

**Implications for forecasting:**

A simple one-row shift would not always represent a 30-minute forecast because consecutive observations may be separated by substantially different time intervals.

We will investigate observation gaps and determine an appropriate timestamp-aware method for constructing future occupancy targets.

The proposed 30-minute prediction horizon remains provisional until its feasibility has been verified.


### 5.2 Observation Gap Analysis

Irregular observation intervals can introduce errors when constructing future occupancy targets.

We examine the frequency of large gaps within each parking facility to determine whether missing reporting periods are isolated events or recurring patterns.

This analysis will help establish acceptable timestamp tolerances for future target construction and prevent observations separated by several hours or days from being treated as consecutive 30-minute measurements.


In [5]:
# Categorize intervals between consecutive observations
intervals = df_clean["TimeDifferenceMinutes"].dropna()

gap_summary = pd.Series({
    "Intervals <= 40 minutes": (intervals <= 40).sum(),
    "Intervals > 40 to 60 minutes": (
        (intervals > 40) & (intervals <= 60)
    ).sum(),
    "Intervals > 60 to 180 minutes": (
        (intervals > 60) & (intervals <= 180)
    ).sum(),
    "Intervals > 180 minutes": (intervals > 180).sum()
})

gap_summary_df = pd.DataFrame({
    "Count": gap_summary,
    "Percentage": (gap_summary / len(intervals) * 100).round(2)
})

display(gap_summary_df)

# Examine the largest observation gaps
print("\nTen largest gaps:")
display(
    df_clean.nlargest(
        10, "TimeDifferenceMinutes"
    )[
        ["SystemCodeNumber", "LastUpdated",
         "TimeDifferenceMinutes"]
    ]
)


,Count,Percentage
Intervals <= 40 minutes,33063,93.21
Intervals > 40 to 60 minutes,390,1.10
Intervals > 60 to 180 minutes,60,0.17
Intervals > 180 minutes,1958,5.52



Ten largest gaps:


,SystemCodeNumber,LastUpdated,TimeDifferenceMinutes
23741,NIA North,2016-10-28 08:16:41,13908.783333
23828,NIA North,2016-11-20 08:01:19,13894.100000
23846,NIA North,2016-11-30 08:02:35,13891.233333
23792,NIA North,2016-11-07 08:00:52,11067.883333
5191,BHMBRCBRG01,2016-10-12 08:04:00,11012.766667
6370,BHMBRCBRG02,2016-10-12 08:04:00,11012.766667
7550,BHMBRCBRG03,2016-10-12 08:04:00,11012.766667
21343,Bull Ring,2016-10-12 08:04:00,11012.766667
15481,BHMNCPNHS01,2016-10-24 07:56:56,6691.883333
23617,NIA Car Parks,2016-12-12 08:02:45,5250.300000


### 5.3 Observation Gap Analysis Findings

Of the 35,471 intervals between consecutive observations within parking facilities:

- **93.21%** are 40 minutes or shorter.
- **1.10%** are between 40 and 60 minutes.
- **0.17%** are between 60 and 180 minutes.
- **5.52%** exceed 180 minutes.

The largest observed gap is approximately 9.66 days, with several substantial gaps occurring at the NIA North facility.

**Implications:**

Most observations occur at intervals broadly consistent with approximately half-hourly reporting. However, the presence of overnight and multi-day gaps makes a simple unconditional row shift inappropriate for constructing forecasting targets.

We will evaluate the number of observations with a subsequent measurement within a defined tolerance of 30 minutes before finalizing the forecasting horizon.

Large gaps will not be bridged through unrestricted interpolation or forward filling.


### 5.4 Feasibility of a 30-Minute Forecasting Horizon

The proposed forecasting objective is to predict parking occupancy approximately 30 minutes into the future.

Because observations are not recorded at perfectly regular intervals, we evaluate how frequently the next observation for a facility occurs within a tolerance of 25–35 minutes.

This analysis estimates the number of potential supervised learning examples and helps determine whether the proposed horizon is practical without introducing targets across long reporting gaps.

In [6]:
# Calculate the interval to the next observation for each facility
df_clean["NextTimestamp"] = (
    df_clean.groupby("SystemCodeNumber")["LastUpdated"].shift(-1)
)

df_clean["MinutesToNext"] = (
    (df_clean["NextTimestamp"] - df_clean["LastUpdated"])
    .dt.total_seconds() / 60
)

# Check how many next observations fall within 25–35 minutes
valid_horizon = df_clean["MinutesToNext"].between(25, 35)

print("Total cleaned observations:", len(df_clean))
print("Observations with a next timestamp:",
      df_clean["MinutesToNext"].notna().sum())

print("Potential 30-minute forecast examples:",
      valid_horizon.sum())

print("Percentage of cleaned observations:",
      round(valid_horizon.mean() * 100, 2), "%")

# Check coverage by parking facility
facility_coverage = (
    df_clean.assign(ValidHorizon=valid_horizon)
    .groupby("SystemCodeNumber")["ValidHorizon"]
    .agg(["sum", "count", "mean"])
)

facility_coverage["mean"] = (
    facility_coverage["mean"] * 100
).round(2)

facility_coverage = facility_coverage.rename(
    columns={
        "sum": "ValidExamples",
        "count": "TotalObservations",
        "mean": "CoveragePercent"
    }
)

display(facility_coverage.sort_values("CoveragePercent"))


Total cleaned observations: 35501
Observations with a next timestamp: 35471
Potential 30-minute forecast examples: 31801
Percentage of cleaned observations: 89.58 %


,ValidExamples,TotalObservations,CoveragePercent
SystemCodeNumber,,,
BHMNCPPLS01,994,1253,79.33
BHMNCPRAN01,979,1174,83.39
BHMNCPNHS01,870,1028,84.63
BHMNCPNST01,1101,1299,84.76
BHMNCPLDH01,1106,1281,86.34
BHMNCPHST01,1149,1301,88.32
BHMBRTARC01,79,88,89.77
BHMBRCBRG02,1061,1180,89.92
NIA North,143,159,89.94


### 5.5 Forecast Horizon Feasibility Findings

The feasibility analysis identified **31,801 potential forecasting examples**, representing **89.58% of the 35,501 cleaned observations**.

These examples have a subsequent observation from the same parking facility occurring between 25 and 35 minutes later.

**Facility-level findings:**

- All 30 parking facilities have observations within the proposed forecasting window.
- Coverage ranges from 79.33% to 91.26%.
- The facility with the lowest coverage, BHMNCPPLS01, retains 994 potential examples.
- Most facilities have approximately 90% coverage.

**Decision:**

We will proceed with an approximately 30-minute forecasting horizon using a 25–35-minute tolerance.

Future occupancy targets will be constructed using the next observation from the same facility only when the elapsed time falls within this window.

This approach avoids treating observations separated by long reporting gaps as valid 30-minute targets.

The final modeling dataset will be constructed after completing the remaining data quality checks. Temporal train/validation/test boundaries will also be enforced to prevent future information from entering training features.


## 6. Capacity and Occupancy Validation

Parking occupancy predictions must respect the physical constraints of each parking facility.

Under the assumption that Capacity represents the maximum number of available parking spaces, valid observations should satisfy:

- Capacity > 0
- Occupancy >= 0
- Occupancy <= Capacity

We examine violations of these constraints, summarize occupancy distributions, and identify whether parking capacity changes over time within individual facilities.

These checks are important because invalid observations could distort model training and produce unrealistic availability predictions.


In [7]:
# Check physical constraints
invalid_capacity = (df_clean["Capacity"] <= 0).sum()
negative_occupancy = (df_clean["Occupancy"] < 0).sum()
over_capacity = (
    df_clean["Occupancy"] > df_clean["Capacity"]
).sum()

print("Invalid capacity (<= 0):", invalid_capacity)
print("Negative occupancy:", negative_occupancy)
print("Occupancy greater than capacity:", over_capacity)

# Summarize capacity and occupancy
print("\nCapacity and occupancy statistics:")
display(df_clean[["Capacity", "Occupancy"]].describe())

# Check whether capacity changes within facilities
capacity_variation = (
    df_clean.groupby("SystemCodeNumber")["Capacity"]
    .nunique()
)

print("\nFacilities with changing capacity:",
      (capacity_variation > 1).sum())

print("\nNumber of distinct capacity values per facility:")
display(capacity_variation.sort_values(ascending=False))


Invalid capacity (<= 0): 0
Negative occupancy: 12
Occupancy greater than capacity: 373

Capacity and occupancy statistics:


,Capacity,Occupancy
count,35501.000000,35501.000000
mean,1399.242613,644.853582
std,1179.939451,657.822376
min,220.000000,-8.000000
25%,500.000000,212.000000
50%,863.000000,448.000000
75%,2009.000000,802.000000
max,4675.000000,4327.000000



Facilities with changing capacity: 0

Number of distinct capacity values per facility:


SystemCodeNumber
BHMBCCMKT01         1
BHMBCCPST01         1
BHMBCCSNH01         1
BHMBCCTHL01         1
BHMBRCBRG01         1
BHMBRCBRG02         1
BHMBRCBRG03         1
BHMBRTARC01         1
BHMEURBRD01         1
BHMEURBRD02         1
BHMMBMMBX01         1
BHMNCPHST01         1
BHMNCPLDH01         1
BHMNCPNHS01         1
BHMNCPNST01         1
BHMNCPPLS01         1
BHMNCPRAN01         1
Broad Street        1
Bull Ring           1
NIA Car Parks       1
NIA North           1
NIA South           1
Others-CCCPS105a    1
Others-CCCPS119a    1
Others-CCCPS133     1
Others-CCCPS135a    1
Others-CCCPS202     1
Others-CCCPS8       1
Others-CCCPS98      1
Shopping            1
Name: Capacity, dtype: int64

### 6.1 Capacity and Occupancy Validation Findings

The dataset contains 35,501 observations after duplicate removal.

**Capacity validation:**
- No zero or negative capacity values were identified.
- Capacity remains constant within each of the 30 parking facilities.

**Occupancy validation:**
- 12 observations contain negative occupancy values.
- 373 observations report occupancy greater than facility capacity.
- The minimum recorded occupancy is -8.
- The maximum recorded occupancy is 4,327.

A total of 385 observations (approximately 1.08%) violate the expected physical occupancy constraints.

**Implications:**

These observations may reflect reporting errors, sensor inaccuracies, or inconsistencies in the source data.

Invalid occupancy values could distort model training and produce unrealistic availability estimates.

Before selecting a cleaning strategy, we will examine how these violations are distributed across facilities and timestamps.

The raw dataset will remain unchanged, and any corrections or exclusions will be documented and implemented reproducibly.


### 6.2 Investigation of Invalid Occupancy Values

To determine an appropriate cleaning strategy, we examine the distribution of invalid occupancy values across parking facilities.

We distinguish between negative occupancy and over-capacity observations and identify whether the violations are isolated or concentrated within particular facilities.

This investigation will guide whether invalid records should be excluded, corrected, or handled through additional validation rules in the automated preprocessing pipeline.


In [8]:
# Identify invalid occupancy records
df_clean["NegativeOccupancy"] = df_clean["Occupancy"] < 0

df_clean["OverCapacity"] = (
    df_clean["Occupancy"] > df_clean["Capacity"]
)

df_clean["InvalidOccupancy"] = (
    df_clean["NegativeOccupancy"] | df_clean["OverCapacity"]
)

# Count violations by facility
invalid_by_facility = (
    df_clean.groupby("SystemCodeNumber")[
        ["NegativeOccupancy", "OverCapacity", "InvalidOccupancy"]
    ]
    .sum()
    .sort_values("InvalidOccupancy", ascending=False)
)

print("Invalid occupancy observations by facility:")
display(invalid_by_facility[invalid_by_facility["InvalidOccupancy"] > 0])

# Inspect examples of invalid observations
print("\nSample invalid observations:")
display(
    df_clean.loc[
        df_clean["InvalidOccupancy"],
        ["SystemCodeNumber", "Capacity", "Occupancy", "LastUpdated"]
    ].head(15)
)


Invalid occupancy observations by facility:


,NegativeOccupancy,OverCapacity,InvalidOccupancy
SystemCodeNumber,,,
BHMBCCTHL01,0,240,240
BHMBRCBRG01,0,86,86
BHMBRCBRG02,0,30,30
NIA North,12,0,12
BHMBCCSNH01,0,11,11
BHMBCCPST01,0,2,2
BHMNCPNHS01,0,2,2
BHMNCPLDH01,0,1,1
BHMMBMMBX01,0,1,1



Sample invalid observations:


,SystemCodeNumber,Capacity,Occupancy,LastUpdated
1391,BHMBCCPST01,317,320,2016-10-08 14:03:38
1608,BHMBCCPST01,317,318,2016-10-22 14:39:47
3533,BHMBCCSNH01,863,865,2016-11-28 11:01:30
3534,BHMBCCSNH01,863,866,2016-11-28 11:32:28
3568,BHMBCCSNH01,863,864,2016-11-30 10:28:38
3617,BHMBCCSNH01,863,869,2016-12-06 07:56:10
3618,BHMBCCSNH01,863,866,2016-12-06 08:29:11
3620,BHMBCCSNH01,863,864,2016-12-06 09:29:14
3621,BHMBCCSNH01,863,868,2016-12-06 10:02:12
3622,BHMBCCSNH01,863,868,2016-12-06 10:29:11


### 6.3 Invalid Occupancy Investigation Findings

A total of 385 observations violate the expected occupancy constraints.

The violations are concentrated in a small number of parking facilities:

- BHMBCCTHL01: 240 over-capacity observations
- BHMBRCBRG01: 86 over-capacity observations
- BHMBRCBRG02: 30 over-capacity observations
- NIA North: 12 negative occupancy observations
- BHMBCSNH01: 11 over-capacity observations
- Four additional facilities: 6 over-capacity observations combined

The three facilities with the most violations account for approximately 92.5% of all invalid observations.

This concentration suggests potential facility-specific reporting inconsistencies, although the underlying causes cannot be confirmed from the available data.

**Cleaning decision:**

Observations with negative occupancy or occupancy exceeding capacity will be excluded from the modeling dataset.

This approach avoids introducing artificial corrections to occupancy measurements.

The original dataset will be preserved, and the number of excluded observations will be recorded as part of the automated data validation process.

After removing invalid observations, the expected dataset size is 35,116 records. Forecasting targets and observation intervals must be recalculated after cleaning.


## 7. Final Data Validation and Cleaning

After identifying duplicate records and invalid occupancy measurements, we construct a validated dataset for subsequent feature engineering and modeling.

The cleaning procedure:
1. Removes exact duplicate observations.
2. Excludes observations with invalid capacity or occupancy values.
3. Preserves valid timestamps and parking facility identifiers.
4. Sorts observations chronologically within each facility.

The original CSV remains unchanged to support reproducibility.

The validated dataset will serve as the input to the forecasting feature engineering pipeline. Observation intervals and future occupancy targets will be recalculated using the validated records.


In [9]:
# Create validated dataset
df_valid = df_clean.loc[
    (df_clean["Capacity"] > 0) &
    (df_clean["Occupancy"] >= 0) &
    (df_clean["Occupancy"] <= df_clean["Capacity"]) &
    (df_clean["LastUpdated"].notna())
].copy()

# Remove temporary audit columns
audit_columns = [
    "TimeDifferenceMinutes",
    "NextTimestamp",
    "MinutesToNext",
    "NegativeOccupancy",
    "OverCapacity",
    "InvalidOccupancy"
]

df_valid = df_valid.drop(
    columns=audit_columns,
    errors="ignore"
)

# Sort chronologically
df_valid = df_valid.sort_values(
    ["SystemCodeNumber", "LastUpdated"]
).reset_index(drop=True)

# Verify final dataset
print("Original observations:", len(df))
print("After duplicate removal:", len(df_clean))
print("Validated observations:", len(df_valid))

print("\nTotal observations removed:",
      len(df) - len(df_valid))

print("\nRemaining duplicate rows:",
      df_valid.duplicated().sum())

print("Remaining invalid occupancy:",
      (
          (df_valid["Occupancy"] < 0) |
          (df_valid["Occupancy"] > df_valid["Capacity"])
      ).sum())

print("\nRemaining parking facilities:",
      df_valid["SystemCodeNumber"].nunique())

display(df_valid.head())


Original observations: 35717
After duplicate removal: 35501
Validated observations: 35116

Total observations removed: 601

Remaining duplicate rows: 0
Remaining invalid occupancy: 0

Remaining parking facilities: 30


,SystemCodeNumber,Capacity,Occupancy,LastUpdated
0,BHMBCCMKT01,577,61,2016-10-04 07:59:42
1,BHMBCCMKT01,577,64,2016-10-04 08:25:42
2,BHMBCCMKT01,577,80,2016-10-04 08:59:42
3,BHMBCCMKT01,577,107,2016-10-04 09:32:46
4,BHMBCCMKT01,577,150,2016-10-04 09:59:48


### 7.1 Final Validation Results

The data validation process successfully produced a cleaned dataset containing **35,116 observations across 30 parking facilities**.

| Validation Metric | Result |
|---|---:|
| Original observations | 35,717 |
| Exact duplicates removed | 216 |
| Invalid occupancy observations removed | 385 |
| Total observations removed | 601 |
| Validated observations | 35,116 |
| Data retained | 98.32% |
| Remaining duplicates | 0 |
| Remaining invalid occupancy values | 0 |
| Parking facilities retained | 30 |

**Key findings:**

- No missing values or invalid timestamps were detected.
- Parking capacity is positive and constant within each facility.
- The original dataset contains negative and over-capacity occupancy measurements.
- Most consecutive observations occur approximately 30 minutes apart.
- Before occupancy filtering, 31,801 observations had a subsequent measurement within 25–35 minutes.

**Forecasting implications:**

An approximately 30-minute forecasting horizon appears feasible. However, valid forecasting pairs must be recalculated after removing invalid observations.

The cleaned dataset will be used to develop chronological training, validation, and test splits and leakage-safe forecasting features.

**Reproducibility:**

The original dataset remains unchanged. Cleaning decisions are documented in this notebook and will be implemented as reusable preprocessing functions for the automated MLOps pipeline.
